# Démonstration Stanza sur manuscrits médiévaux français (corpus CATMuS)

**Notebook Colab — Module NLP — Master Data/IA — MD5 Volet 2 — 2026**

Ce notebook complète le script `stanza_medieval_demo.py` fourni séparément.
Il couvre :

- **Partie 1** : reprise rapide de la démonstration du pipeline Stanza (identique au script `.py`, à exécuter ici pour profiter du GPU Colab)
- **Partie 2** : **fine-tuning effectif** du tagger POS/morphologique de Stanza sur le modèle `fro`, ce qui nécessite de cloner le dépôt source `stanfordnlp/stanza` et de lancer l'entraînement via les scripts shell du dépôt — **Stanza ne permet PAS d'entraîner un modèle via l'API Pipeline Python**.

**Avant de commencer** : Menu *Exécution* → *Modifier le type d'exécution* → sélectionner **GPU (T4)**.


## Partie 0 — Installation des dépendances

In [ ]:
!pip install -q stanza datasets

import stanza
print("Stanza version :", stanza.__version__)

import torch
print("GPU disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))


## Partie 1 — Pipeline Stanza sur le corpus médiéval (rappel)

Cette partie reprend le contenu du script `stanza_medieval_demo.py`.
Le détail de chaque concept (pipeline, tokenisation, POS, lemmatisation,
dépendances, NER) est expliqué dans le script ; ce notebook se concentre
sur l'exécution avec GPU et l'enchaînement vers la Partie 2.


### 1.1 — Chargement du corpus CATMuS (HuggingFace datasets)

In [ ]:
from datasets import load_dataset

FRENCH_MEDIEVAL_PATTERNS = [
    "old french", "ancien français", "ancien francais",
    "middle french", "moyen français", "moyen francais",
    "french",
]

def is_french_medieval(language_value: str) -> bool:
    if not language_value:
        return False
    lv = language_value.strip().lower()
    return any(p in lv for p in FRENCH_MEDIEVAL_PATTERNS)

def load_catmus_sample(n_lines: int = 12, split: str = "train"):
    print(f"Téléchargement de CATMuS/medieval (split={split})…")
    ds = load_dataset("CATMuS/medieval", split=split, streaming=True)
    selected = []
    for row in ds:
        if is_french_medieval(row.get("language", "")):
            text = row["text"].strip()
            if 20 <= len(text) <= 200:
                selected.append(text)
        if len(selected) >= n_lines:
            break
    return selected

corpus = load_catmus_sample(n_lines=5)
for i, line in enumerate(corpus, 1):
    print(f"{i}. {line}")


**Note** : si la cellule ci-dessus ne retourne aucune ligne, c'est que les
valeurs exactes de la colonne `language` diffèrent de celles supposées
dans `FRENCH_MEDIEVAL_PATTERNS`. Exécutez la cellule de diagnostic
suivante pour les identifier précisément.


In [ ]:
# Cellule de diagnostic — à exécuter seulement si load_catmus_sample()
# ne retourne aucune ligne ci-dessus.
ds_small = load_dataset("CATMuS/medieval", split="train", streaming=True)
sample_languages = set()
for i, row in enumerate(ds_small):
    sample_languages.add(row.get("language", "?"))
    if i > 2000:
        break
print("Valeurs de `language` observées sur 2000 lignes :")
print(sorted(sample_languages))


### 1.2 — Construction du pipeline `fro` et annotation

In [ ]:
stanza.download('fro')

nlp_fro = stanza.Pipeline(
    lang='fro',
    processors='tokenize,pos,lemma,depparse',
    use_gpu=torch.cuda.is_available(),
    verbose=False,
)

text = " ".join(corpus) if corpus else (
    "Artus li rois fu mout vaillanz et mout redotez de toutes genz. "
    "Lancelot vint en Bretaigne pour querre la roine Guenievre."
)
doc = nlp_fro(text)

# Note : upos/lemma peuvent être None sur certains tokens (notamment la
# ponctuation, ou si le tagger ne produit pas d'étiquette) — on protège
# l'affichage avec `or "_"` pour éviter un TypeError sur le format-spec.
for sent in doc.sentences:
    for word in sent.words:
        upos = word.upos or "_"
        lemma = word.lemma or "_"
        print(f"{word.text:<15} upos={upos:<6} lemma={lemma:<15} "
              f"head={word.head:<3} deprel={word.deprel}")
    print()


Pour le détail pédagogique de chaque étape (tokenisation, POS, lemmatisation,
dépendances, dépendances inter-phrases, NER par contournement + comparaison
avec `fr`), reportez-vous aux fonctions correspondantes dans
`stanza_medieval_demo.py`, qui peuvent aussi être exécutées ici :


In [ ]:
# Si vous avez uploadé stanza_medieval_demo.py dans ce notebook Colab
# (panneau Fichiers à gauche), vous pouvez réutiliser toutes ses fonctions :
#
# from stanza_medieval_demo import (
#     demo_tokenization, demo_pos_morphology, demo_lemmatization,
#     demo_dependency_parsing, demo_cross_sentence_dependencies,
#     demo_ner_workaround_fro, demo_ner_comparison_fr,
# )
# demo_tokenization(doc)
# demo_pos_morphology(doc)
# ... etc.
#
# En particulier, demo_lemmatization(doc) affiche aussi un rapport de
# couverture du lemmatiseur (proportion de tokens pour lesquels Stanza
# ne produit aucun lemme). Ce phénomène est normal pour fro : le treebank
# d'entraînement (Profiterole, ~227 000 tokens) est petit et couvre une
# graphie médiévale très variable sur quatre siècles, ce qui laisse
# le dictionnaire du lemmatiseur clairsemé face à un nouveau texte.


---
## Partie 2 — Fine-tuning effectif du tagger POS (GPU requis)

**Rappel important** : Stanza n'expose aucune fonction d'entraînement via
l'API `stanza.Pipeline`. L'entraînement de tout processeur neuronal
(tokenizer, MWT, tagger POS, lemmatiseur, parser de dépendances, NER)
nécessite de **cloner le dépôt source** `stanfordnlp/stanza` et d'invoquer
les scripts d'entraînement (`stanza.utils.training.run_pos`, etc.) en
ligne de commande. C'est ce que cette partie met en œuvre, étape par étape.

Le processeur ciblé est le **tagger POS/morphologique** (UPOS, XPOS, UFeats),
conformément au périmètre fixé pour cette démonstration.


### 2.1 — Préparation des données (rappel, voir aussi le script .py)

In [ ]:
import os, random
from collections import Counter

random.seed(42)

# Corpus non étiqueté de démonstration (en conditions réelles : un extrait
# CATMuS filtré et débarrassé de toute annotation existante).
UNLABELED_CORPUS_DEMO = [
    "Artus li rois fu mout vaillanz et mout redotez de toutes genz.",
    "Lancelot vint en Bretaigne pour querre la roine Guenievre.",
    "Gauvain et Perceval chevauchierent ensemble vers Camaalot.",
    "Merlin parla au roi de la queste qui devoit estre faite.",
    "La pucele estoit mout bele et mout cortoise en son langage.",
    "Li chevaliers prist son escu et sa lance pour aler au tournoi.",
    "Tristan ama Yseut plus que nul autre chevalier n'ama dame.",
    "Charlemagne tint son empire dis et set anz sans guerre perdre.",
]

print(f"{len(UNLABELED_CORPUS_DEMO)} phrases non étiquetées chargées.")
print("\nATTENTION : ce volume est UNIQUEMENT démonstratif. Un fine-tuning")
print("sérieux requiert plusieurs centaines à quelques milliers de phrases.")


In [ ]:
def prepare_finetuning_data(raw_sentences, nlp, output_dir="./pos_finetuning_data",
                             train_ratio=0.7, dev_ratio=0.15, seed=42):
    """Auto-annotation (silver standard) + conversion CoNLL-U + split.
    Voir stanza_medieval_demo.py pour la version commentée en détail."""
    os.makedirs(output_dir, exist_ok=True)
    random.seed(seed)

    print("[1/4] Auto-annotation des phrases (silver standard)...")
    print("  ATTENTION : ces annotations contiennent les erreurs du modèle fro")
    print("  actuel. Une relecture humaine serait nécessaire en conditions réelles.")
    annotated = []
    for raw_text in raw_sentences:
        doc = nlp(raw_text)
        for sent in doc.sentences:
            tokens = [{
                "id": w.id, "text": w.text, "lemma": w.lemma or "_",
                "upos": w.upos or "_", "xpos": w.xpos or "_",
                "feats": w.feats or "_", "head": w.head, "deprel": w.deprel or "_",
            } for w in sent.words]
            annotated.append({"text": sent.text, "tokens": tokens})
    print(f"  {len(annotated)} phrases auto-annotées.")

    print("[2/4] Split train/dev/test...")
    indices = list(range(len(annotated)))
    random.shuffle(indices)
    n = len(indices)
    n_train = max(1, int(n * train_ratio))
    n_dev = max(1, int(n * dev_ratio))
    splits = {
        "train": [annotated[i] for i in indices[:n_train]],
        "dev":   [annotated[i] for i in indices[n_train:n_train + n_dev]],
        "test":  [annotated[i] for i in indices[n_train + n_dev:]] or [annotated[indices[-1]]],
    }
    for name, sents in splits.items():
        print(f"  {name:<6}: {len(sents)} phrases")

    print("[3/4] Écriture des fichiers CoNLL-U...")
    shorthand = "fro_custom"
    paths = {}
    for name, sents in splits.items():
        path = os.path.join(output_dir, f"{shorthand}-ud-{name}.conllu")
        with open(path, "w", encoding="utf-8") as f:
            for i, sent in enumerate(sents, 1):
                f.write(f"# sent_id = {name}_{i:04d}\n")
                f.write(f"# text = {sent['text']}\n")
                for tok in sent["tokens"]:
                    f.write("\t".join([
                        str(tok["id"]), tok["text"], tok["lemma"], tok["upos"],
                        tok["xpos"], tok["feats"], str(tok["head"]),
                        tok["deprel"], "_", "_",
                    ]) + "\n")
                f.write("\n")
        paths[name] = path
        print(f"  Écrit : {path}")

    print("[4/4] Rapport de couverture...")
    all_tokens = [t["text"] for s in annotated for t in s["tokens"]]
    all_upos = [t["upos"] for s in annotated for t in s["tokens"]]
    report = {
        "n_sentences": len(annotated), "n_tokens": len(all_tokens),
        "vocab_size": len(set(t.lower() for t in all_tokens)),
        "upos_distribution": dict(Counter(all_upos)),
    }
    print(f"  {report}")
    return {"paths": paths, "report": report, "shorthand": shorthand}


# Chemin ABSOLU, volontairement : la cellule suivante clone le dépôt
# stanza et fait %cd vers /content/stanza_src, ce qui changerait la
# résolution d'un chemin relatif comme "./pos_finetuning_data". En
# fixant un chemin absolu ici, result["paths"] reste valide quel que
# soit le répertoire de travail courant au moment où on le relit.
result = prepare_finetuning_data(
    UNLABELED_CORPUS_DEMO, nlp_fro,
    output_dir="/content/pos_finetuning_data",
)


### 2.2 — Clonage du dépôt source `stanfordnlp/stanza`

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/stanfordnlp/stanza.git stanza_src
%cd stanza_src

# Installer Stanza en mode développement depuis les sources clonées
# (nécessaire pour que les scripts stanza.utils.* utilisent CE dépôt,
# pas la version installée via pip plus haut)
!pip install -q -e .


### 2.3 — Convention de nommage UD attendue par les scripts

Les scripts `stanza.utils.datasets.prepare_pos_treebank` attendent une
arborescence au format `$UDBASE/{corpus}/{corpus_short}-ud-{train,dev,test}.conllu`,
où `{corpus}` suit la convention `UD_Langue-NomDuTreebank` (par exemple
`UD_English-EWT`). Notre corpus personnalisé est donc placé sous un nom
fictif `UD_Old_French-Custom`, avec le short name `fro_custom` déjà utilisé
dans `prepare_finetuning_data()`.


In [ ]:
import os, shutil

UDBASE = "/content/ud_data"
CORPUS_NAME = "UD_Old_French-Custom"
SHORT_NAME = result["shorthand"]  # "fro_custom", défini en Partie 2.1

corpus_dir = os.path.join(UDBASE, CORPUS_NAME)
os.makedirs(corpus_dir, exist_ok=True)

for split, src_path in result["paths"].items():
    dst_path = os.path.join(corpus_dir, f"{SHORT_NAME}-ud-{split}.conllu")
    shutil.copy(src_path, dst_path)
    print(f"Copié : {dst_path}")

os.environ["UDBASE"] = UDBASE
print(f"\nUDBASE = {UDBASE}")
print(f"Contenu de {corpus_dir} :")
print(os.listdir(corpus_dir))


### 2.4 — Variables d'environnement requises par les scripts d'entraînement

In [ ]:
import os

# DATA_ROOT : racine des fichiers intermédiaires générés par les scripts
DATA_ROOT = "/content/stanza_data_root"
os.makedirs(DATA_ROOT, exist_ok=True)
os.environ["DATA_ROOT"] = DATA_ROOT
os.environ["POS_DATA_DIR"] = os.path.join(DATA_ROOT, "pos")
os.makedirs(os.environ["POS_DATA_DIR"], exist_ok=True)

# WORDVEC_DIR : vecteurs de mots pré-entraînés (on réutilise le .pt
# téléchargé pour fro via stanza.download, plus simple que de
# télécharger des vecteurs fastText séparés pour cette démonstration)
WORDVEC_DIR = "/content/extern_wordvec"
os.makedirs(WORDVEC_DIR, exist_ok=True)
os.environ["WORDVEC_DIR"] = WORDVEC_DIR

# Chemin du fichier de pretrain existant pour fro
PRETRAIN_PATH = os.path.expanduser(
    "~/stanza_resources/fro/pretrain/conll17.pt"
)
print("Fichier pretrain attendu :", PRETRAIN_PATH)
print("Existe :", os.path.exists(PRETRAIN_PATH))

for var in ["UDBASE", "DATA_ROOT", "POS_DATA_DIR", "WORDVEC_DIR"]:
    print(f"{var} = {os.environ.get(var)}")


### 2.5 — Conversion des données au format interne Stanza

`prepare_pos_treebank` lit les fichiers `.conllu` sous `$UDBASE/{corpus}/`
et produit les fichiers intermédiaires attendus par `run_pos` sous
`$POS_DATA_DIR`.


In [ ]:
!python3 -m stanza.utils.datasets.prepare_pos_treebank {CORPUS_NAME}


### 2.6 — Lancement de l'entraînement effectif du tagger POS

`--max_steps` est volontairement réduit pour cette démonstration (le
corpus de 8 phrases ne justifie pas un entraînement long). En conditions
réelles avec plusieurs centaines de phrases, retirez ou augmentez cette
limite et laissez l'early-stopping sur le dev set faire son travail.

`--wordvec_pretrain_file` pointe vers les vecteurs pré-entraînés du
modèle `fro` existant, téléchargés via `stanza.download('fro')` en
Partie 1 — cela évite de devoir entraîner des embeddings depuis zéro.


In [ ]:
!python3 -m stanza.utils.training.run_pos {SHORT_NAME} \
    --wordvec_pretrain_file {PRETRAIN_PATH} \
    --max_steps 200 \
    --save_dir /content/saved_models_finetuned


**En cas d'erreur "no such file" sur le pretrain** : le chemin exact du
fichier `.pt` téléchargé par `stanza.download('fro')` peut varier selon
la version de Stanza. Listez le contenu de `~/stanza_resources/fro/pretrain/`
pour retrouver le nom exact, ou relancez la commande avec `--no_pretrain`
(performance dégradée mais fonctionnel, cf. la documentation officielle
qui indique que `run_pos.py` supporte ce flag pour les langues sans
vecteurs disponibles).


### 2.7 — Évaluation du modèle fine-tuné sur le split test

In [ ]:
!python3 -m stanza.utils.training.run_pos {SHORT_NAME} \
    --wordvec_pretrain_file {PRETRAIN_PATH} \
    --save_dir /content/saved_models_finetuned \
    --score_test


### 2.8 — Utilisation du modèle fine-tuné dans un pipeline Stanza

Une fois entraîné, le modèle se charge comme n'importe quel modèle Stanza
en pointant explicitement vers le fichier `.pt` produit par l'entraînement,
via les arguments `{processor}_model_path` du `Pipeline`.


In [ ]:
import glob

finetuned_models = glob.glob("/content/saved_models_finetuned/pos/*.pt")
print("Modèles fine-tunés trouvés :", finetuned_models)

if finetuned_models:
    nlp_finetuned = stanza.Pipeline(
        lang="fro",
        processors="tokenize,pos,lemma,depparse",
        pos_model_path=finetuned_models[0],
        verbose=False,
    )
    doc_test = nlp_finetuned("Artus li rois fu mout vaillanz.")
    for sent in doc_test.sentences:
        for word in sent.words:
            print(f"{word.text:<15} upos={word.upos}  xpos={word.xpos}  feats={word.feats}")
else:
    print("Aucun modèle trouvé — vérifiez les étapes précédentes.")


---
## Conclusion

Ce notebook a démontré :

1. L'exécution du pipeline Stanza complet sur un extrait du corpus médiéval CATMuS, avec le modèle `fro` (tokenisation, POS, lemmatisation, dépendances)
2. Le contournement NER par gazetier pour `fro` et la comparaison avec le modèle `fr` qui dispose d'un NER officiel
3. La préparation de données non étiquetées pour le fine-tuning (auto-annotation, conversion CoNLL-U, split train/dev/test)
4. Le fine-tuning effectif du tagger POS via clonage du dépôt source et scripts shell, conformément à la contrainte de Stanza qui n'expose aucune API d'entraînement Python

**Limite à retenir** : le corpus de démonstration (8 phrases) est strictement pédagogique. Un fine-tuning utile en conditions réelles nécessiterait un corpus CATMuS filtré beaucoup plus large, idéalement avec une relecture humaine au moins partielle des annotations silver standard avant l'entraînement.
